# Day 86: EAGLE Speculative Decoding

**Layer:** Advanced


## Concept Overview

EAGLE (Extrapolation Algorithm for Greater Language-model Efficiency) drafts tokens at the feature level rather than token level, using a lightweight head that predicts the next hidden state instead of the next token distribution.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. EAGLE Draft Mechanism

EAGLE uses a 1-layer autoregressive head operating on hidden states. This is faster than a full draft model and achieves higher acceptance rates.


In [ ]:
import torch, numpy as np

class EAGLEHead(torch.nn.Module):
    """Simplified EAGLE draft head: 1-layer transformer on hidden states."""
    def __init__(self, d_model=4096, vocab_size=32000):
        super().__init__()
        self.fc = torch.nn.Linear(d_model * 2, d_model)
        self.head = torch.nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, hidden, prev_hidden):
        # Concatenate current and previous hidden state
        x = torch.cat([hidden, prev_hidden], dim=-1)
        x = torch.nn.functional.relu(self.fc(x))
        return self.head(x)

d_model, vocab = 4096, 32000
head = EAGLEHead(d_model, vocab)
total_params = sum(p.numel() for p in head.parameters())
print(f"EAGLE head parameters: {total_params/1e6:.1f}M")
print(f"Compare to 7B model: {7000/total_params:.0f}x smaller")
print()
# Simulate acceptance rates
np.random.seed(42)
print("EAGLE vs standard speculative decoding:")
for name, base_accept in [("Standard (7B->70B)", 0.75), ("EAGLE (head->70B)", 0.82)]:
    gamma = 4  # draft tokens
    accepted = [base_accept**i * (1-base_accept if i<gamma else base_accept**gamma) for i in range(gamma+1)]
    expected_tokens = sum((i+1)*p for i,p in enumerate(accepted))
    speedup = expected_tokens / 1.0
    print(f"  {name}: expected_tokens={expected_tokens:.2f} speedup={speedup:.2f}x")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- EAGLE (Extrapolation Algorithm for Greater Language-model Efficiency) drafts tokens at the feature level rather than token level, using a lightweight head that predicts the next hidden state instead of the next token distribution..
- EAGLE (Extrapolation Algorithm for Greater Language-model Efficiency) drafts tok....
- Day 86 implementation complete.
